[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Objetos de *Seaborn*.

Las funciones de *Seaborn* (*notebook* 32) son fáciles de usar, pero cada una corresponde a un tipo de gráfico fijo. La interfaz `seaborn.objects`, introducida en la versión 0.12 (septiembre de 2022), adopta un enfoque distinto: la **gramática de gráficos por capas**, en la que un gráfico se construye combinando componentes independientes.

## La gramática de gráficos por capas.

La *gramática de gráficos* fue propuesta por *Leland Wilkinson* (1999) y refinada como *gramática de gráficos por capas* (*layered grammar of graphics*) por [*Hadley Wickham*](https://hadley.nz/), quien la implementó en [*ggplot2*](https://ggplot2.tidyverse.org/) para *R*. Describe un gráfico como la combinación de componentes independientes:

| Componente | Descripción | En `seaborn.objects` |
|---|---|---|
| **Datos** | Conjunto de datos que se desea visualizar. | `so.Plot(data, ...)` |
| **Mapeos estéticos** | Asignación de variables a propiedades visuales: posición, color, tamaño, forma… | `x=`, `y=`, `color=`, `pointsize=`, ... |
| **Objetos geométricos** | Representación visual: puntos, líneas, barras, áreas… | Marcas: `so.Dot()`, `so.Line()`, `so.Bar()`, ... |
| **Transformaciones estadísticas** | Cálculos aplicados a los datos antes de graficarlos. | `so.Hist()`, `so.Agg()`, `so.PolyFit()`, ... |
| **Posiciones** | Acomodo de los elementos que se enciman. | Movimientos: `so.Dodge()`, `so.Stack()`, `so.Jitter()` |
| **Escalas** | Traducción de los valores a ejes y colores. | `.scale()` |
| **Facetas** | División del gráfico en paneles según una o más variables. | `.facet()`, `.pair()` |
| **Sistema de coordenadas** | Espacio en el que se dibuja el gráfico (cartesiano, polar, etc.). | Sólo cartesiano |

<img src="img/grammar_of_graphics.png">

La implementación de referencia de esta gramática es *ggplot2*; en *Python*, [*plotnine*](https://plotnine.org/) la reproduce con casi la misma sintaxis y se estudia en el tema extra del *notebook* 40. La interfaz `seaborn.objects` aplica los mismos principios con un conjunto de funciones más reducido.

**Nota:** La documentación de *Seaborn* aún describe `seaborn.objects` como una interfaz **experimental e incompleta**: es estable para un uso serio, pero le faltan funciones (por ejemplo, otros sistemas de coordenadas) y puede tener cambios incompatibles en versiones futuras. Las funciones del *notebook* 32 siguen siendo la interfaz más usada.

Por convención, `seaborn.objects` se importa como `so`. En adelante, se seguirá dicha convención.

https://seaborn.pydata.org/tutorial/objects_interface.html

* La siguiente celda importa `seaborn` con el alias `sns` y el submódulo `objects` con el alias `so`.

In [ ]:
import seaborn as sns
from seaborn import objects as so

## Conjuntos de datos.

Los ejemplos de esta notebook utilizan tres conjuntos de datos clásicos que *Seaborn* incluye y que pueden cargarse con `sns.load_dataset()`.

### `"iris"`.

El conjunto de datos [*Iris*](https://en.wikipedia.org/wiki/Iris_flower_data_set), introducido por el estadístico *Ronald Fisher* en 1936, contiene mediciones morfológicas de 150 flores de tres especies de iris.

| Columna | Tipo | Descripción |
|---|---|---|
| `sepal_length` | `float` | Longitud del sépalo en cm |
| `sepal_width` | `float` | Ancho del sépalo en cm |
| `petal_length` | `float` | Longitud del pétalo en cm |
| `petal_width` | `float` | Ancho del pétalo en cm |
| `species` | `category` | Especie: `setosa`, `versicolor` o `virginica` |

Es ideal para ilustrar diagramas de dispersión, distribuciones y facetas por categoría.

### `"tips"`.

El conjunto de datos *Tips* recoge 244 registros de propinas en un restaurante, con información sobre la cuenta, la propina y características de los comensales.

| Columna | Tipo | Descripción |
|---|---|---|
| `total_bill` | `float` | Importe total de la cuenta en dólares |
| `tip` | `float` | Propina en dólares |
| `sex` | `category` | Sexo del pagador: `Male` o `Female` |
| `smoker` | `category` | Si había fumadores en la mesa: `Yes` o `No` |
| `day` | `category` | Día de la semana: `Thur`, `Fri`, `Sat` o `Sun` |
| `time` | `category` | Turno: `Lunch` o `Dinner` |
| `size` | `int` | Número de comensales |

Es útil para visualizaciones de conteos por categoría, comparaciones de grupos y movimientos de posición.

### `"flights"`.

El conjunto de datos *Flights* registra el número mensual de pasajeros de una aerolínea entre 1949 y 1960, con 144 registros en total.

| Columna | Tipo | Descripción |
|---|---|---|
| `year` | `int` | Año de la observación (1949–1960) |
| `month` | `category` | Mes abreviado en inglés |
| `passengers` | `int` | Número de pasajeros en ese mes |

Es el conjunto de referencia para series temporales y trayectorias ordenadas.

## La clase ```so.Plot```.

Todo gráfico de esta interfaz es un objeto de la clase ```so.Plot```, que define los datos y los mapeos estéticos comunes a todas las capas.

```
so.Plot(data=<datos>, x=<x>, y=<y>, color=<color>, <otros mapeos>)
```

Donde:

* ```<datos>``` es un *dataset* compatible con un *dataframe* de *Pandas*.
* ```<x>``` y ```<y>``` son los identificadores de las columnas que se asignarán a los ejes $x$ y $y$.
* ```<color>``` es opcional: el identificador de la columna que define el color.
* ```<otros mapeos>``` son otras propiedades visuales que se pueden asignar a columnas, como `pointsize`, `marker`, `alpha`, `linewidth` o `linestyle`.

Los objetos ```so.Plot``` tienen métodos para agregar cada componente de la gramática: `.add()` (capas con marcas, estadísticas y movimientos), `.facet()` y `.pair()` (facetas), `.scale()` (escalas), `.label()`, `.limit()`, `.layout()` y `.theme()` (presentación) y `.save()` (guardado).

https://seaborn.pydata.org/generated/seaborn.objects.Plot.html

## Métodos en cascada.

Los métodos de ```so.Plot``` regresan un **nuevo** objeto ```so.Plot```, por lo que se encadenan uno tras otro. Para escribir la cadena en varias líneas, se encierra entre paréntesis. El gráfico sólo se dibuja al mostrarse o guardarse, así que un objeto se puede guardar en una variable y reutilizar como base de varios gráficos.

### El método ```so.Plot.add()```.

Este método agrega una **capa** al gráfico. Cada capa tiene una marca (objeto geométrico) y, opcionalmente, transformaciones estadísticas y movimientos.

```
<plot>.add(<mark>, <transformaciones>, <mapeos de la capa>)
```

Donde:

* ```<mark>``` es una marca de ```seaborn.objects``` (p. ej. ```so.Dot()```, ```so.Bar()```, ```so.Line()```). Sus argumentos **fijan** propiedades para todos los elementos, por ejemplo `so.Dot(color="red")`.
* ```<transformaciones>``` son opcionales: estadísticas (p. ej. ```so.Hist()```, ```so.Agg()```) y movimientos (p. ej. ```so.Dodge()```), que se aplican en orden.
* ```<mapeos de la capa>``` son opcionales: mapeos que sólo aplican a esta capa, como `color="species"`. Con `color=None` la capa ignora el mapeo de color definido en `so.Plot()`.

https://seaborn.pydata.org/generated/seaborn.objects.Plot.add.html

**Ejemplo:**

* La siguiente celda carga el conjunto de datos "iris" y lo asigna a la variable `dataset`, mostrando su contenido.

In [ ]:
dataset = sns.load_dataset("iris")
dataset

* La siguiente celda inicializa un objeto `so.Plot` asignando las variables para los ejes $x$ e $y$, pero sin definir ninguna representación gráfica aún.

In [ ]:
so.Plot(data=dataset,
        x="sepal_length",
        y="sepal_width")

* La siguiente celda añade una capa de puntos (`so.Dot`) al gráfico para visualizar los datos.

In [ ]:
(so.Plot(data=dataset,
         x="sepal_length",
         y="sepal_width")
 .add(so.Dot()))

* La siguiente celda asigna la variable `"species"` al color de los puntos, permitiendo diferenciar las categorías.

In [ ]:
(so.Plot(data=dataset,
         x="sepal_length",
         y="sepal_width",
         color="species")
 .add(so.Dot()))

* La siguiente celda utiliza el método `.facet()` para crear subgráficos separados por la categoría `"species"`.

In [ ]:
(so.Plot(data=dataset,
         x='sepal_length',
         y='sepal_width')
 .facet('species')
 .add(so.Dot()))

* La siguiente celda combina una marca de barras (`so.Bars`) con una transformación estadística de histograma (`so.Hist`), creando histogramas de la longitud del sépalo, uno por especie.

In [ ]:
(so.Plot(data=dataset,
         x='sepal_length',
         color='species')
 .facet('species')
 .add(so.Bars(),
      so.Hist()))

* La siguiente celda carga el *dataset* `tips` y aplica una escala logarítmica a ambos ejes con el método `.scale()`. La escala logarítmica separa los puntos de las cuentas pequeñas, que en escala lineal quedan amontonados.

In [ ]:
tips = sns.load_dataset("tips")

(so.Plot(data=tips,
         x="total_bill",
         y="tip")
 .add(so.Dots())
 .scale(x="log", y="log"))

### Combinación de capas.

Es posible añadir múltiples capas a un mismo gráfico utilizando llamadas sucesivas al método `.add()`.

* La siguiente celda muestra cómo combinar múltiples capas, añadiendo una estimación de densidad de kernel (KDE) representada como área y línea.

In [ ]:
(so.Plot(data=dataset,
         x="sepal_length",
         color="species")
 .add(so.Area(),
      so.KDE())
 .add(so.Line(),
      so.KDE()))

## Marcas (*Marks*).

Las marcas definen la forma geométrica con la que se representan los datos. Las principales marcas de `seaborn.objects` son:

| Marca | Descripción |
|-------|-------------|
| `so.Dot` | Un punto por observación. |
| `so.Dots` | Puntos pensados para muchas observaciones: por omisión son huecos y semitransparentes, para que se distingan cuando se enciman. |
| `so.Line` | Línea que une los puntos ordenados por $x$. |
| `so.Lines` | Como `so.Line`, pero más eficiente cuando se trazan muchas líneas. |
| `so.Path` | Línea que une los puntos **en el orden de los datos**, sin ordenarlos por $x$. |
| `so.Bar` | Barras, con estilo pensado para pocas barras (bordes marcados). |
| `so.Bars` | Barras pensadas para muchas barras contiguas, como en un histograma. |
| `so.Area` | Área rellena bajo una línea. |
| `so.Band` | Banda rellena entre dos valores, por ejemplo un intervalo. |
| `so.Range` | Segmento entre dos valores, por ejemplo una barra de error. |
| `so.Dash` | Guion corto en cada valor. |
| `so.Text` | Texto en cada posición. |

Cualquier marca puede mapear propiedades visuales (color, tamaño, transparencia, ...) a columnas.

https://seaborn.pydata.org/api.html#mark-objects

### `so.Dots`.

La marca `so.Dots` es la variante de `so.Dot` pensada para muchas observaciones: los puntos son huecos y semitransparentes por omisión. El ejemplo siguiente además asigna una tercera variable al tamaño de los puntos con el mapeo `pointsize`, lo que también puede hacerse con `so.Dot`.

```
so.Plot(data=<dataset>, x=<x>, y=<y>, pointsize=<col>).add(so.Dots())
```

Donde:

* `<dataset>` es el conjunto de datos que contiene las columnas a utilizar.
* `<x>` es la columna que se asignará al eje $x$.
* `<y>` es la columna que se asignará al eje $y$.
* `<col>` es la columna numérica cuyo valor determinará el tamaño de cada punto.

https://seaborn.pydata.org/generated/seaborn.objects.Dots.html

* La siguiente celda crea un diagrama de dispersión con `so.Dots`, codificando la longitud del pétalo en el tamaño de cada punto mediante el parámetro estético `pointsize`.

In [ ]:
(so.Plot(data=dataset,
         x="sepal_length",
         y="sepal_width",
         color="species",
         pointsize="petal_length")
 .add(so.Dots()))

### `so.Bars`.

A diferencia de `so.Bar`, `so.Bars` está pensado para muchas barras contiguas, como las de un histograma calculado con `so.Hist`: dibuja bordes delgados para que las barras no se vean separadas.

```
so.Plot(data=<dataset>, x=<x_column>).add(so.Bars(), so.Hist())
```

Donde:

* `<dataset>` es el conjunto de datos que contiene la columna a utilizar.
* `<x_column>` es la columna que se asignará al eje $x$ para calcular el histograma.

https://seaborn.pydata.org/generated/seaborn.objects.Bars.html

* La siguiente celda muestra un ejemplo de cómo utilizar `so.Bars` junto con `so.Hist` para crear un histograma de la longitud del sépalo (`sepal_length`) por especie del conjunto de datos "iris".

In [ ]:
(so.Plot(data=dataset,
         x="sepal_length",
         color="species")
 .add(so.Bars(), so.Hist(binwidth=0.3)))

### `so.Path` y `so.Line`.

`so.Line` ordena los puntos por el valor de $x$ antes de unirlos; `so.Path` los une en el orden en que aparecen en los datos. La diferencia importa cuando el orden de los datos tiene significado propio, por ejemplo el tiempo en un diagrama de dispersión conectado.

* La siguiente celda carga el *dataset* `flights` y construye una tabla con los pasajeros de cada mes en 1959 y en 1960.

In [ ]:
flights = sns.load_dataset("flights")

meses = flights.pivot(index="month", columns="year", values="passengers").reset_index()
meses.columns = [str(c) for c in meses.columns]
meses = meses[["month", "1959", "1960"]]
meses

* La siguiente celda une los meses en orden de enero a diciembre con `so.Path` y etiqueta cada punto con `so.Text`. La trayectoria muestra el ciclo anual: los pasajeros suben hacia el verano y bajan en otoño.

In [ ]:
(so.Plot(data=meses,
         x="1959",
         y="1960",
         text="month")
 .add(so.Path(marker="o"))
 .add(so.Text(halign="left", offset=6))
 .label(x="Pasajeros en 1959 (miles)", y="Pasajeros en 1960 (miles)"))

* Con `so.Line`, los mismos puntos se unen en orden de $x$ y el ciclo anual se pierde.

In [ ]:
(so.Plot(data=meses,
         x="1959",
         y="1960")
 .add(so.Line(marker="o")))

## Transformaciones estadísticas y de movimiento.

Las transformaciones modifican los datos antes de dibujarlos. Se dividen en dos tipos:

* **Estadísticas**: calculan resúmenes o agregaciones.
    * `so.Hist`: histograma.
    * `so.KDE`: estimación de densidad.
    * `so.Count`: conteo de observaciones.
    * `so.Agg`: agregación (promedio por omisión) de `y` por cada valor de `x`.
    * `so.Est`: agregación con un intervalo (por ejemplo, de confianza), para usarse con `so.Range` o `so.Band`.
    * `so.PolyFit`: ajuste de un polinomio (una recta con `order=1`).
* **Movimientos** (`so.Dodge`, `so.Stack`, `so.Jitter`, `so.Shift`): reposicionan los elementos sin alterar sus valores.

https://seaborn.pydata.org/api.html#stat-objects

### `so.Count`.

La transformación `so.Count` cuenta el número de observaciones en cada categoría y pasa el resultado como la altura de la barra u otro canal geométrico.

```
so.Plot(data=<dataset>, x=<x_column>).add(
    so.Bar(), so.Count())
```
Donde:
* `<dataset>` es el conjunto de datos que contiene la columna a utilizar.
* `<x_column>` es la columna que se asignará al eje $x$ para contar las observaciones por categoría.

https://seaborn.pydata.org/generated/seaborn.objects.Count.html

* La siguiente celda muestra un ejemplo de cómo utilizar `so.Count` junto con `so.Bar` para crear un diagrama de barras que cuenta las observaciones por día (`day`) y sexo (`sex`) del *dataset* `tips`. Sin un movimiento, las barras de ambos sexos se dibujan en la misma posición y se enciman.

In [ ]:
(so.Plot(data=tips,
         x="day",
         color="sex")
 .add(so.Bar(), so.Count()))

### `so.Dodge`.

El movimiento `so.Dodge` separa lateralmente los elementos que se solaparían, permitiendo comparar grupos dentro de una misma categoría.

https://seaborn.pydata.org/generated/seaborn.objects.Dodge.html

* La siguiente celda muestra un ejemplo de cómo utilizar `so.Dodge` para separar lateralmente las barras de cada sexo (`sex`) dentro de cada día (`day`) del conjunto de datos "tips".

In [ ]:
(so.Plot(data=tips,
         x="day",
         color="sex")
 .add(so.Bar(), so.Count(), so.Dodge()))

### `so.Stack`.

El movimiento `so.Stack` apila los valores de los diferentes grupos a lo largo del eje de respuesta, mostrando tanto los valores individuales como el total acumulado.

```
so.Plot(data=<dataset>, x=<x>, color=<grupo>).add(so.Bar(), so.Count(), so.Stack())
```

Donde:
* `<dataset>` es el conjunto de datos que contiene las columnas a utilizar.
* `<x>` es la columna categórica que se asignará al eje $x$.
* `<grupo>` es la columna que se utilizará para diferenciar los grupos por color.

https://seaborn.pydata.org/generated/seaborn.objects.Stack.html

* La siguiente celda muestra un ejemplo de cómo utilizar `so.Stack` junto con `so.Bar` y `so.Count` para apilar las barras por sexo (`sex`) dentro de cada día (`day`) del conjunto de datos "tips".

In [ ]:
(so.Plot(data=tips,
         x="day",
         color="sex")
 .add(so.Bar(), so.Count(), so.Stack()))

### `so.Jitter`.

El movimiento `so.Jitter` desplaza los puntos de forma aleatoria para reducir el solapamiento en datos categóricos. El parámetro `width` controla la amplitud del desplazamiento horizontal.
```
so.Plot(data=<dataset>, x=<x_column>, y=<y_column>).add(
    so.Dot(), so.Jitter(width=<jitter_width>))
```
Donde:
* `<dataset>` es el conjunto de datos que contiene las columnas a utilizar.
* `<x_column>` es la columna que se asignará al eje $x$.
* `<y_column>` es la columna que se asignará al eje $y$.
* `<jitter_width>` es un valor numérico que determina la cantidad de desplazamiento horizontal aplicado a los puntos para reducir el solapamiento.

https://seaborn.pydata.org/generated/seaborn.objects.Jitter.html

* La siguiente celda muestra un ejemplo de cómo utilizar `so.Jitter` para desplazar aleatoriamente los puntos y reducir el solapamiento en los datos de longitud del sépalo (`sepal_length`) por especie del conjunto de datos "iris".

In [ ]:
(so.Plot(data=dataset,
         x="species",
         y="sepal_length",
         color="species")
 .add(so.Dot(), so.Jitter(width=0.3)))

### `so.Agg`, `so.Est` y `so.PolyFit`.

Una misma gráfica puede combinar capas con distintas estadísticas. Los siguientes ejemplos construyen, capa por capa, gráficos que con las funciones de *Seaborn* requerirían opciones especiales o no serían posibles.

* La siguiente celda muestra el intervalo de confianza del 95 % de la propina por día y sexo (`so.Range` con `so.Est`) y encima la propina promedio (`so.Dot` con `so.Agg`). Las capas se dibujan en el orden en que se agregan, por eso el intervalo va primero. Ambas capas se separan con `so.Dodge()`.

In [ ]:
(so.Plot(data=tips,
         x="day",
         y="tip",
         color="sex")
 .add(so.Range(), so.Est(errorbar="ci"), so.Dodge())
 .add(so.Dot(pointsize=8), so.Agg(), so.Dodge()))

* La siguiente celda combina los puntos individuales (`so.Dots`) con una recta de regresión (`so.Line` con `so.PolyFit`). La recta usa `color=None` para ignorar el mapeo de color y ajustarse con todos los datos.

In [ ]:
(so.Plot(data=tips,
         x="total_bill",
         y="tip",
         color="time")
 .add(so.Dots())
 .add(so.Line(color="black"), so.PolyFit(order=1), color=None))

## Métodos de configuración de `so.Plot`.

Además de `.add()`, `.facet()` y `.scale()`, los objetos de `so.Plot` cuentan con otros métodos para controlar la presentación del gráfico.

https://seaborn.pydata.org/generated/seaborn.objects.Plot.html

### El método `.label()`.

El método `.label()` permite personalizar el título de la figura y las etiquetas de los ejes y de la leyenda.

```
so.Plot(data=<dataset>, x=<x_column>, y=<y_column>, color=<group_column>).add(
    so.Bars(), so.Hist(), so.Stack()).label(
        title=<title>, x=<x_label>, y=<y_label>, color=<legend_title>)
```
Donde:
* `<dataset>` es el conjunto de datos que contiene las columnas a utilizar.
* `<x_column>` es la columna que se asignará al eje $x$.
* `<y_column>` es la columna que se asignará al eje $y$.
* `<group_column>` es la columna que se utilizará para diferenciar los grupos por color.
* `<title>` es una cadena de texto que se utilizará como título del gráfico.
* `<x_label>` es una cadena de texto que se utilizará como etiqueta del eje $x$.
* `<y_label>` es una cadena de texto que se utilizará como etiqueta del eje $y$.
* `<legend_title>` es una cadena de texto que se utilizará como título de la leyenda para el canal de color.


https://seaborn.pydata.org/generated/seaborn.objects.Plot.label.html

* La siguiente celda muestra un ejemplo de cómo utilizar `.label()` para añadir un título y etiquetas descriptivas a los ejes y a la leyenda de un diagrama de dispersión del conjunto de datos "iris".

In [ ]:
(so.Plot(data=dataset,
         x="sepal_length",
         y="sepal_width",
         color="species")
 .add(so.Dot())
 .label(title="Dimensiones del sépalo por especie",
        x="Longitud del sépalo (cm)",
        y="Ancho del sépalo (cm)",
        color="Especie"))

### El método `.limit()`.

El método `.limit()` establece los límites de los ejes $x$ e $y$. Sólo cambia la **vista**: las estadísticas se calculan con todos los datos, como `coord_cartesian()` en *plotnine* (ver el *notebook* 40).

```
so.Plot(data=<dataset>, x=<x_column>, y=<y_column>).add(
    <mark>()).limit(x=(<x_min>, <x_max>), y=(<y_min>, <y_max>))
```

Donde:

* `<x_min>` es el valor mínimo del eje $x$.
* `<x_max>` es el valor máximo del eje $x$.
* `<y_min>` es el valor mínimo del eje $y$.
* `<y_max>` es el valor máximo del eje $y$.

https://seaborn.pydata.org/generated/seaborn.objects.Plot.limit.html

* La siguiente celda muestra un ejemplo de cómo utilizar `.limit()` para restringir el rango visible de los ejes de un diagrama de dispersión del conjunto de datos "iris".

In [ ]:
(so.Plot(data=dataset,
         x="sepal_length",
         y="sepal_width",
         color="species")
 .add(so.Dot())
 .limit(x=(4.5, 8), y=(2, 4.5)))

### El método `.layout()`.

El método `.layout()` controla el tamaño de la figura generada.

```
so.Plot(data=<dataset>, x=<x_column>, y=<y_column>).add(
    <mark>()).layout(size=(<ancho>, <alto>))
```

Donde:

* `<ancho>` y `<alto>` son las dimensiones de la figura expresadas en pulgadas.

https://seaborn.pydata.org/generated/seaborn.objects.Plot.layout.html

* La siguiente celda muestra un ejemplo de cómo utilizar `.layout()` para establecer el tamaño de la figura facetada del conjunto de datos "iris".

In [ ]:
(so.Plot(data=dataset,
         x="sepal_length",
         y="sepal_width",
         color="species")
 .facet("species")
 .add(so.Dot())
 .layout(size=(10, 4)))

### Los métodos `.scale()` y `.theme()`.

`.scale()` también define la paleta de un mapeo de color, y `.theme()` cambia la apariencia general del gráfico con un diccionario de parámetros de *Matplotlib*; las funciones `sns.axes_style()` y `sns.plotting_context()` del *notebook* 32 producen esos diccionarios.

* La siguiente celda usa la paleta cualitativa `"Set2"` y el estilo `"whitegrid"`.

In [ ]:
(so.Plot(data=dataset,
         x="sepal_length",
         y="sepal_width",
         color="species")
 .add(so.Dot())
 .scale(color="Set2")
 .theme(sns.axes_style("whitegrid")))

### El método `.pair()`.

`.pair()` traza el mismo gráfico para varias variables, un panel por variable, compartiendo el otro eje.

* La siguiente celda compara la longitud del sépalo y del pétalo por especie.

In [ ]:
(so.Plot(data=dataset,
         y="species",
         color="species")
 .pair(x=["sepal_length", "petal_length"])
 .add(so.Dots(), so.Jitter(0.5)))

### El método `.save()`.

`.save(<ruta>, dpi=<resolución>)` guarda el gráfico en un archivo; el formato se deduce de la extensión.

* La siguiente celda guarda un gráfico en `data_tmp/iris_sepalo.png`.

In [ ]:
import os

os.makedirs("data_tmp", exist_ok=True)
(so.Plot(data=dataset, x="sepal_length", y="sepal_width", color="species")
 .add(so.Dot())
 .save(os.path.join("data_tmp", "iris_sepalo.png"), dpi=150))
os.path.exists(os.path.join("data_tmp", "iris_sepalo.png"))

## De `seaborn.objects` a la gramática completa.

Lo estudiado en este *notebook* es la gramática de gráficos: datos, mapeos, capas con marcas y estadísticas, movimientos, escalas y facetas. Esos mismos componentes existen, con más opciones, en *ggplot2* y en *plotnine*:

| Componente | `seaborn.objects` | *plotnine* |
| :--- | :--- | :--- |
| Datos y mapeos | `so.Plot(df, x=..., y=..., color=...)` | `ggplot(df, aes(x=..., y=..., color=...))` |
| Capa | `.add(so.Dot())` | `+ geom_point()` |
| Estadística | `.add(so.Bars(), so.Hist())` | `+ geom_histogram()` o `+ stat_bin()` |
| Agregación con intervalo | `.add(so.Range(), so.Est())` | `+ stat_summary(fun_data='mean_cl_normal')` |
| Posición | `so.Dodge()`, `so.Stack()`, `so.Jitter()` | `position='dodge'`, `'stack'`, `'jitter'` |
| Escalas | `.scale(x="log", color="Set2")` | `+ scale_x_log10()`, `+ scale_color_brewer(...)` |
| Facetas | `.facet(col=..., row=...)` | `+ facet_grid('fila ~ columna')` |
| Límites de la vista | `.limit(x=...)` | `+ coord_cartesian(xlim=...)` |
| Coordenadas | Sólo cartesianas | `coord_flip()`, `coord_cartesian()`, ... |
| Títulos | `.label(title=..., x=...)` | `+ labs(title=..., x=...)` |
| Apariencia | `.theme(...)` | `+ theme_*()`, `+ theme(...)` |

El tema extra del *notebook* 40 estudia la gramática a fondo con *plotnine*.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>